In [39]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config
import ijson
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


In [40]:
# object_name = "AmazonEC2.json"
object_name = "AmazonTimestream.json"

TARGET_RECORDS = 3000 
sample_products = []

response = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Το ijson διαβάζει κατευθείαν από το stream byte-byte
    parser = ijson.kvitems(response, 'products')
    
    count = 0
    for sku, product_data in parser:
        #Mε την προοπτική να δημιουργεί πεδίο με sku με την αντίστοιχη τιμή μέσα στο dict αλλά αυτό ήδη υπάρχει
        # product_data['sku'] = sku
        sample_products.append(product_data)
        
        count += 1
        if count >= TARGET_RECORDS:
            break
            
    print(f"Downloaded  {len(sample_products)} records μέσω streaming.")

finally:
    response.close()
    response.release_conn()

# Μετατροπή σε αρχικό DataFrame
df_products = pd.json_normalize(sample_products)
print(f"DataFrame: Rows = {df_products.shape[0]}, Columns = {df_products.shape[1]}")
df_products.head()

Downloaded  973 records μέσω streaming.
DataFrame: Rows = 973, Columns = 25


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.instanceType,attributes.vcpu,attributes.memory,attributes.storage,attributes.networkPerformance,attributes.engineCode,attributes.databaseEngine,attributes.licenseModel,attributes.deploymentOption,attributes.usagetype,attributes.operation,attributes.normalizationSizeFactor,attributes.regionCode,attributes.servicename,attributes.description,attributes.storageMedia,attributes.volumeType,attributes.minVolumeSize,attributes.maxVolumeSize,attributes.disableactivationconfirmationemail
0,9M9RM284G374U245,Database Instance,AmazonTimestream,Asia Pacific (Mumbai),AWS Region,db.influx.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,1,InfluxDB,No license required,Multi-AZ,APS3-MultiAZUsage-Db.influx.24xlarge,Compute:001,384,ap-south-1,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
1,9R7TG9RQ5D8QJSZ5,Database Instance,AmazonTimestream,Asia Pacific (Sydney),AWS Region,db.influxIOIncluded.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,2,InfluxDB,No license required,Cluster,APS2-ClusterNodeUsage-Db.influxIOIncluded.24xl...,Compute:002,192,ap-southeast-2,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
2,ZXQNH69MUXTTEUCD,Influx Optimized Storage,AmazonTimestream,US East (N. Virginia),AWS Region,NaN,NaN,NaN,NaN,NaN,1,InfluxDB,NaN,Single-AZ,USE1-SingleAZ-InfluxDBStorage-InfluxIOIncludedT3,Storage:001,NaN,us-east-1,Amazon Timestream,SingleAZ Influx IOPS Included (16K IOPS),SSD,Influx IOPS Included with 16K IOPS,400 GB,16 TB,NaN
3,Z9UMVX66CBK745V7,Database Instance,AmazonTimestream,Asia Pacific (Sydney),AWS Region,db.influxIOIncluded.12xlarge,48,384 GiB,InfluxDb Compatible,Up to 20 Gigabit,2,InfluxDB,No license required,Cluster,APS2-ClusterNodeUsage-Db.influxIOIncluded.12xl...,Compute:002,96,ap-southeast-2,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN
4,A66WQEVBSFUP6U3T,Database Instance,AmazonTimestream,EU (Milan),AWS Region,db.influx.24xlarge,96,768 GiB,InfluxDb Compatible,Up to 40 Gigabit,1,InfluxDB,No license required,Cluster,EUS1-ClusterNodeUsage-Db.influx.24xlarge,Compute:001,192,eu-south-1,Amazon Timestream,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
# Στο πάνω κελί είχα μία λίστα η οποία είχε μέσα n sku, μαζί με όλα τα attributes τουσ.
# Τώρα στο βήμα αυτό κάνω access την λίστα και απομονώνω σε ένα set μόνο τον κωδικό των n sku (η επιλογή set βασίζεται στην γρήγορη αναζήτηση)
target_skus = {p['sku'] for p in sample_products}

# Αυτή θα είναι η αντίστοιχη sample products του πάνω βήματος. Θα κρατάει τα ζευγάρια sku, με στοιχεία πληρωμής, και μετά θα την κάνουμε dataframe
terms_list = []

parser = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Πάμε βαθύτερα και από το λεξικό terms θα στοχεύσουμε μόνο στις On-Demand υπηρεσίες.
    terms_parser = ijson.kvitems(parser, 'terms.OnDemand')

    # Προφανώς δεν τα θέλω όλα!!Μόνο εκείνα των οποίων το sku βρίσκεται στο set το οποίο δημιούργησα
    for sku, term_offers in terms_parser:
        if sku not in target_skus:
            continue
        
        # Αποθηκεύουμε το sku και ολόκληρο το raw λεξικό των terms του
        terms_list.append({
            'skuNew': sku,
            'termsOndemand': term_offers
        })
        
        # Aπλά για να βεβαιωθώ ότι όσα products πήρα άλλες τόσες και οι τιμές 
        if len(terms_list) >= len(target_skus):
            break
            
    print(f"Downloaded {len(terms_list)} matching terms μέσω streaming.")

finally:
    parser.close()
    parser.release_conn()


df_terms = pd.DataFrame(terms_list)
print(f"Terms DataFrame: Rows = {df_terms.shape[0]}, Columns = {df_terms.shape[1]}")
df_terms.head()

Downloaded 973 matching terms μέσω streaming.
Terms DataFrame: Rows = 973, Columns = 2


,skuNew,termsOndemand
0,9M9RM284G374U245,{'9M9RM284G374U245.JRTCKXETXF': {'offerTermCod...
1,9R7TG9RQ5D8QJSZ5,{'9R7TG9RQ5D8QJSZ5.JRTCKXETXF': {'offerTermCod...
2,ZXQNH69MUXTTEUCD,{'ZXQNH69MUXTTEUCD.JRTCKXETXF': {'offerTermCod...
3,Z9UMVX66CBK745V7,{'Z9UMVX66CBK745V7.JRTCKXETXF': {'offerTermCod...
4,A66WQEVBSFUP6U3T,{'A66WQEVBSFUP6U3T.JRTCKXETXF': {'offerTermCod...


Θα δουλέψουμε αρχικά με το 2ο dataframe το οποίο περιέχει τα δεδομένα τιμολόγησης. Κρίνονται απαραίτητες 4 ενέργειες
- Άνοιγμα 2η στήλης και άπλωμα δεδομένων
- Αντιστοιχία εσωτερικού sku με αυτό που έβαλα εγώ, και πέταμα μίας στήλης εκ των 2
- Μελέτη για εντοπισμό καθολικών στηλών 
- Αφαίρεση περιττών στηλών
- Κατανόηση pricing και αντιστοίχιση με τους άλλους παρόχους

In [42]:
# Πάιρνω την πρώτη εγγραφή προκειμένου να κάνω έναν έλεγχο των πεδίων
sample_row = df_terms.iloc[0]
print("SKU:", sample_row['skuNew'])

raw_dict = sample_row['termsOndemand']

# Βρίσκουμε το κλειδί (το σύνθετο hash, π.χ. SKU.OfferTermCode)
offer_hash_key = list(raw_dict.keys())[0]
offer_content = raw_dict[offer_hash_key]

print("\n--- Περιεχόμενα προσφοράς (Offer Content) ---")
for k, v in offer_content.items():
    # if k != 'priceDimensions':
    print(f"{k}: {v}")

SKU: 9M9RM284G374U245

--- Περιεχόμενα προσφοράς (Offer Content) ---
offerTermCode: JRTCKXETXF
sku: 9M9RM284G374U245
effectiveDate: 2026-02-01T00:00:00Z
priceDimensions: {'9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7': {'rateCode': '9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7', 'description': '$26.112 per Instance-Hour for Multi-AZ db.influx.24xlarge Compute in Asia Pacific (Mumbai)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'Hrs', 'pricePerUnit': {'USD': '26.1120000000'}, 'appliesTo': []}}
termAttributes: {}


Ο παρακάτω κώδικας υλοποιεί ένα μέρος του πρώτου βήματος. Πετάει ένα περιττό hash το οποίο ήταν sku + offerCode, και ανοίγει εν μέρη το λεξικό termsOnDemand το οποίο κατασκεύσαμε όταν πήραμε τα δεδομένα και τα μετατρέψαμε σε datframe. Συγκεκριμένα εξάγει μερικά πεδία και τα κάνει κανονικές στήλες. Το απευθείας normalize δοκιμάστηκε και απετύχε, οπότε και προχωρήσαμε με ένα for loop.

In [43]:
flattened_list = []

for idx, row in df_terms.iterrows():
    skuDefaultValue = row['skuNew']
    raw_dict = row['termsOndemand']
    
    # ΜΠετάω το αρχικό κλειδί το οποίο είχε το dict. Περισσοτερα στην αναφορά
    for hash_key, offer_content in raw_dict.items():
        
        # Εξάγω τα πεδία ένα - ένα και φτιάχνω μία δική μου δομή πιο υύκολη στην ανάλυση
        item = {
            'skuNew': skuDefaultValue,
            'offerTermCode': offer_content.get('offerTermCode'),
            'sku': offer_content.get('sku'),
            'effectiveDate': offer_content.get('effectiveDate'),
            'termAttributes': offer_content.get('termAttributes'),
            'priceDimensions': offer_content.get('priceDimensions') # Το αφήνουμε λεξικό!
        }
        flattened_list.append(item)

df_terms = pd.DataFrame(flattened_list)

print(df_terms.columns)
df_terms.head(2)

Index(['skuNew', 'offerTermCode', 'sku', 'effectiveDate', 'termAttributes',
       'priceDimensions'],
      dtype='object')


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,9M9RM284G374U245,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,{},{'9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7': {'r...
1,9R7TG9RQ5D8QJSZ5,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,{},{'9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7': {'r...


Επόμενο βήμα είναι το περαιτέρω άνοιγμα του λεξικού το οποίο κρύβει μέσα τις πληροφοριες τιμολόγησης. Συγκεκριμένα το priceDimensions. Όπως φαίνεται και από το πάνω αποτέλεσμα του κελιού, μέσα στο λεξικό αυτό υπάρχει άλλο ένα περίεργο hash - κλειδί, το οποίο όμως όπως και στο παραπάνω κελί θα απορρίψουμε. Ο κώδικας λοιπόν ακολουθεί την ίδια τακτική: διατρέχει γραμμή γραμμή, προσπερνάει το περίεργο αυτό, και τραβάει μόνο τα πραγματικά δεδομένα.

In [44]:
df_terms.head()

,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,9M9RM284G374U245,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,{},{'9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7': {'r...
1,9R7TG9RQ5D8QJSZ5,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,{},{'9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7': {'r...
2,ZXQNH69MUXTTEUCD,JRTCKXETXF,ZXQNH69MUXTTEUCD,2026-02-01T00:00:00Z,{},{'ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7': {'r...
3,Z9UMVX66CBK745V7,JRTCKXETXF,Z9UMVX66CBK745V7,2026-02-01T00:00:00Z,{},{'Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7': {'r...
4,A66WQEVBSFUP6U3T,JRTCKXETXF,A66WQEVBSFUP6U3T,2026-02-01T00:00:00Z,{},{'A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7': {'r...


In [45]:
itemsList = []

for idx, row in df_terms.iterrows():

    # Κατασκευάζουμε εξ'ολοκλήρου νεό dataframe. Δεν κάνουμε ενέργειες πάνω στον υπάρχον. Οπότε φτιάχνουμε γραμμή γραμμή με τα πεδία που έχουμε. Για αυτό και τα εξάξουμε ένα - ένα
    skuNew = row['skuNew']
    offerTermCode = row['offerTermCode']
    skuDefaultValue = row['sku']
    effectiveDate = row['effectiveDate']
    termAttributes = row['termAttributes']
    
    # Παίρνουμε το λεξικό του priceDimensions
    priceDimensionDict = row['priceDimensions']

    # Κοιτάζει εάν όντως το priceDimensions είναι λεξικό. Αν δεν είναι δεν μπαίνει καν μέσα στο loop. Έτσι και δεν κρασάρει, και αποφεύγω να δημιουργήσω γραμμές στις οποίες τα δεδομένα είναι ελλιπή
    if isinstance(priceDimensionDict, dict):
        # ΔΌπως και στο πάνω κελί, με τον τρόπο αυτό αγνοούμε το εσωτετικό xxx.xxx.xxx
        for dimensionsDict_hash_key, dimensionsDict_content in priceDimensionDict.items():
            
            # Το pricePerUnit είναι και αυτό με την σειρά του λεξικού οποτε πριν εφαρμόσω την μέθοδο get προσέχω για να βεβαιωθώ ότι το βρήκα και δεν έπεσα στην περίπτωση "κακών" δεδομένων
            price_per_unit_dict = dimensionsDict_content.get('pricePerUnit', {})
            usd_price = price_per_unit_dict.get('USD') if isinstance(price_per_unit_dict, dict) else None
            
            item = {
                'skuNew': skuNew,
                'offerTermCode': offerTermCode,
                'sku': skuDefaultValue,
                'effectiveDate': effectiveDate,
                'termAttributes': termAttributes,
                'rateCode': dimensionsDict_content.get('rateCode'),
                'description': dimensionsDict_content.get('description'),
                'beginRange': dimensionsDict_content.get('beginRange'),
                'endRange': dimensionsDict_content.get('endRange'),
                'unit': dimensionsDict_content.get('unit'),
                'priceUSD': usd_price,   # Aυτό το πεδίο μέσω του ελέγχου που κάναμε πιο πάνω, ή θα είναι None ή θα έχει κάποια τιμή. Οπότε θα το χρησιμοποιήσω μετά για έλεγχω
                'appliesTo': dimensionsDict_content.get('appliesTo')
            }
            itemsList.append(item)

df_terms_final = pd.DataFrame(itemsList)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (973, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,9M9RM284G374U245,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,{},9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7,$26.112 per Instance-Hour for Multi-AZ db.infl...,0,Inf,Hrs,26.1120000000,[]
1,9R7TG9RQ5D8QJSZ5,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,{},9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7,$14.9952 per Instance-Hour for Cluster db.infl...,0,Inf,Hrs,14.9952000000,[]
2,ZXQNH69MUXTTEUCD,JRTCKXETXF,ZXQNH69MUXTTEUCD,2026-02-01T00:00:00Z,{},ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7,USD 0.50 per GB-Month for SingleAZ InfluxIOInc...,0,Inf,GB-Mo,0.5000000000,[]
3,Z9UMVX66CBK745V7,JRTCKXETXF,Z9UMVX66CBK745V7,2026-02-01T00:00:00Z,{},Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7,$7.4976 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,7.4976000000,[]
4,A66WQEVBSFUP6U3T,JRTCKXETXF,A66WQEVBSFUP6U3T,2026-02-01T00:00:00Z,{},A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7,$13.248 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,13.2480000000,[]


##### Επεξεργασία πίνακα με τα δεδομένα τιμολόγησης 
1. Λίστα appliesTo ή οποία μπορεί να φαίνεται κενή, αλλά θα την φροντίσουμε με explode και reindex για κάθε ενδεχόμενο. Επίσης το το πεδίο termAtrributes το οποίο είναι ένα dict κενό, θα το αφαιρέσουμε καθώς μετά από μελέτη του documentaion κρίθηκε άχρηστο αφ'ης στιγμής κρατάμε μόνο On-Demand εγγραφές Δεν χρειάζονται περίεργα loop ή εντολές σύνθετες. Απλή χρήση των 2 εντολών. 

In [46]:
df_terms_final = df_terms_final.explode('appliesTo').reset_index(drop=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (973, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,9M9RM284G374U245,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,{},9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7,$26.112 per Instance-Hour for Multi-AZ db.infl...,0,Inf,Hrs,26.1120000000,NaN
1,9R7TG9RQ5D8QJSZ5,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,{},9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7,$14.9952 per Instance-Hour for Cluster db.infl...,0,Inf,Hrs,14.9952000000,NaN
2,ZXQNH69MUXTTEUCD,JRTCKXETXF,ZXQNH69MUXTTEUCD,2026-02-01T00:00:00Z,{},ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7,USD 0.50 per GB-Month for SingleAZ InfluxIOInc...,0,Inf,GB-Mo,0.5000000000,NaN
3,Z9UMVX66CBK745V7,JRTCKXETXF,Z9UMVX66CBK745V7,2026-02-01T00:00:00Z,{},Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7,$7.4976 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,7.4976000000,NaN
4,A66WQEVBSFUP6U3T,JRTCKXETXF,A66WQEVBSFUP6U3T,2026-02-01T00:00:00Z,{},A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7,$13.248 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,13.2480000000,NaN


In [47]:
if 'termAttributes' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['termAttributes'])

print (f"Dimension (rows,cols): {df_terms_final.shape}")
df_terms_final.head()


Dimension (rows,cols): (973, 11)


,skuNew,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,9M9RM284G374U245,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7,$26.112 per Instance-Hour for Multi-AZ db.infl...,0,Inf,Hrs,26.1120000000,NaN
1,9R7TG9RQ5D8QJSZ5,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7,$14.9952 per Instance-Hour for Cluster db.infl...,0,Inf,Hrs,14.9952000000,NaN
2,ZXQNH69MUXTTEUCD,JRTCKXETXF,ZXQNH69MUXTTEUCD,2026-02-01T00:00:00Z,ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7,USD 0.50 per GB-Month for SingleAZ InfluxIOInc...,0,Inf,GB-Mo,0.5000000000,NaN
3,Z9UMVX66CBK745V7,JRTCKXETXF,Z9UMVX66CBK745V7,2026-02-01T00:00:00Z,Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7,$7.4976 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,7.4976000000,NaN
4,A66WQEVBSFUP6U3T,JRTCKXETXF,A66WQEVBSFUP6U3T,2026-02-01T00:00:00Z,A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7,$13.248 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,13.2480000000,NaN


2. Αρχικά παρατηρούμε 2 στήλες με το sku της υπηρεσίας. Η μία ήταν εξαρχής μέσα στα δεδομένα τιμολόγησης με την ονομασία sku, και η άλλη προστέθηκε κατά το άνοιγμα των υπηρεσιών. Συγκεκριμένα όλα τα δεδομένα terms είχαν σαν αρχικό αναγνωριστικό το sku χύμα, και μετά τα δεδομένα: κάπως έτσι "xxx: {dict with pricing info}". Οπότε το αρχικό κλειδί το κάναμε στήλη. Τώρα θα γράψουμε κώδικα ο οποίος ελέγχει αν υπάρχει ταύτιση skuNew με sku, αν δεν υπάρχει θα πετάει την εγγραφή, και στο τέλος θα αφαιρεί μία από τις δύο στήλες.

In [50]:
rowCount = len(df_terms_final)

#Βάζω το if για να μπορώ να τρέχω το κελί και μόνο του χωρίς να πετάει error
if 'skuNew' in df_terms_final.columns and 'sku' in df_terms_final.columns:

    # Φτιάχνω την συνθήκη ελέγχου - διαγραφής μιας υπηρεσίας και την εφαρμόζω απευθείας μετά πάνω στο dataframe. Γλιτώνω το loop 
    condition = (df_terms_final['skuNew'] == df_terms_final['sku'])

    df_filtered_terms = df_terms_final[condition]

    df_terms_final = df_filtered_terms.copy()

print(f"Initial records: {rowCount}")
print(f"Rejected records: {rowCount - len(df_terms_final)}")

if 'skuNew' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['skuNew'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Initial records: 973
Rejected records: 0
Dimensions (rows,cols): (973, 10)


,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,9M9RM284G374U245,2026-02-01T00:00:00Z,9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7,$26.112 per Instance-Hour for Multi-AZ db.infl...,0,Inf,Hrs,26.1120000000,NaN
1,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,2026-02-01T00:00:00Z,9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7,$14.9952 per Instance-Hour for Cluster db.infl...,0,Inf,Hrs,14.9952000000,NaN
2,JRTCKXETXF,ZXQNH69MUXTTEUCD,2026-02-01T00:00:00Z,ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7,USD 0.50 per GB-Month for SingleAZ InfluxIOInc...,0,Inf,GB-Mo,0.5000000000,NaN
3,JRTCKXETXF,Z9UMVX66CBK745V7,2026-02-01T00:00:00Z,Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7,$7.4976 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,7.4976000000,NaN
4,JRTCKXETXF,A66WQEVBSFUP6U3T,2026-02-01T00:00:00Z,A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7,$13.248 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,13.2480000000,NaN


3. Η στήλη effectiveDate είναι σε ίδιο μήκος κύματος με τις στήλες που έχει η azure, και η google στα δεδομένα της. Περιγράφει την ημερομηνία και ώρα εκκίνησης της συγκεκριμένης τιμής που υπάρχει για την υπηρεσία. Στην εργασία δεν μας ενδιαφέρει η ιστορικότητα των δεδομέων, οπότε την αφαιρούμε απευθείας με τις αντίστοιχες εντολές.

In [52]:
if 'effectiveDate' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['effectiveDate'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (973, 9)


,offerTermCode,sku,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,9M9RM284G374U245,9M9RM284G374U245.JRTCKXETXF.6YS6EN2CT7,$26.112 per Instance-Hour for Multi-AZ db.infl...,0,Inf,Hrs,26.1120000000,NaN
1,JRTCKXETXF,9R7TG9RQ5D8QJSZ5,9R7TG9RQ5D8QJSZ5.JRTCKXETXF.6YS6EN2CT7,$14.9952 per Instance-Hour for Cluster db.infl...,0,Inf,Hrs,14.9952000000,NaN
2,JRTCKXETXF,ZXQNH69MUXTTEUCD,ZXQNH69MUXTTEUCD.JRTCKXETXF.6YS6EN2CT7,USD 0.50 per GB-Month for SingleAZ InfluxIOInc...,0,Inf,GB-Mo,0.5000000000,NaN
3,JRTCKXETXF,Z9UMVX66CBK745V7,Z9UMVX66CBK745V7.JRTCKXETXF.6YS6EN2CT7,$7.4976 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,7.4976000000,NaN
4,JRTCKXETXF,A66WQEVBSFUP6U3T,A66WQEVBSFUP6U3T.JRTCKXETXF.6YS6EN2CT7,$13.248 per Instance-Hour for Cluster db.influ...,0,Inf,Hrs,13.2480000000,NaN
